# FinançaSimples - Versão 1: Gerenciamento de Finanças Pessoais (POO, SQLite)

**Trabalho Final da Disciplina: Desenvolvimento Rápido de Aplicações em Python**

**Autor:** Manus (Assistente de Programação)

## ⚠️ ATENÇÃO: Limitação do Google Colab

O código abaixo implementa a interface gráfica (GUI) usando **Tkinter**, conforme solicitado no documento da atividade.

**O Google Colab não suporta interfaces gráficas interativas (GUI) baseadas em Tkinter diretamente no navegador.**

**O código foi unificado em um bloco para testar a lógica central (POO e SQLite) no Colab.** Para usar a interface gráfica completa, você deve copiar o código para um arquivo `.py` e executá-lo em seu computador local (Windows, macOS, Linux).

O bloco de código finaliza com um **Teste de Lógica** para comprovar a criação do banco de dados, a inserção de dados e o cálculo do saldo, atendendo aos requisitos de POO, SQLite e consultas parametrizadas.

In [ ]:
import sqlite3
import tkinter as tk
from tkinter import ttk, messagebox
from datetime import datetime

# --- Classe DBManager (Gerenciador de Banco de Dados) ---

class DBManager:
    """
    Gerenciador de Banco de Dados (SQLite) para o FinançaSimples.
    Responsável por toda a persistência de dados, garantindo o uso de consultas parametrizadas.
    """
    def __init__(self, db_path='financas.db'):
        self.db_path = db_path
        self._conectar_e_criar_tabelas()

    def _conectar_e_criar_tabelas(self):
        """Conecta ao banco de dados e cria as tabelas se não existirem."""
        try:
            conn = sqlite3.connect(self.db_path)
            cursor = conn.cursor()

            # Tabela categorias
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS categorias (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    nome TEXT UNIQUE NOT NULL,
                    tipo TEXT NOT NULL CHECK(tipo IN ('RECEITA', 'DESPESA'))
                );
            """)

            # Tabela contas
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS contas (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    nome TEXT UNIQUE NOT NULL,
                    saldo_inicial REAL NOT NULL
                );
            """)

            # Tabela transacoes
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS transacoes (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    id_conta INTEGER,
                    id_categoria INTEGER,
                    tipo TEXT NOT NULL CHECK(tipo IN ('RECEITA', 'DESPESA')),
                    valor REAL NOT NULL,
                    data TEXT,
                    descricao TEXT,
                    FOREIGN KEY (id_conta) REFERENCES contas(id),
                    FOREIGN KEY (id_categoria) REFERENCES categorias(id)
                );
            """)

            conn.commit()
            conn.close()
            # Inserir dados iniciais para facilitar o uso
            self._inserir_dados_iniciais()

        except sqlite3.Error as e:
            print(f"Erro ao inicializar o banco de dados: {e}")

    def _inserir_dados_iniciais(self):
        """Insere categorias e contas padrão se o banco estiver vazio."""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()

        # Inserir Categorias Padrão
        categorias_padrao = [
            ("Salário", "RECEITA"),
            ("Investimentos", "RECEITA"),
            ("Aluguel", "DESPESA"),
            ("Alimentação", "DESPESA"),
            ("Transporte", "DESPESA"),
            ("Lazer", "DESPESA"),
        ]
        try:
            cursor.executemany("INSERT INTO categorias (nome, tipo) VALUES (?, ?)", categorias_padrao)
            conn.commit()
        except sqlite3.IntegrityError:
            # Ignora se já existirem
            pass

        # Inserir Contas Padrão
        contas_padrao = [
            ("Conta Corrente", 1000.00),
            ("Poupança", 5000.00),
            ("Carteira", 50.00),
        ]
        try:
            cursor.executemany("INSERT INTO contas (nome, saldo_inicial) VALUES (?, ?)", contas_padrao)
            conn.commit()
        except sqlite3.IntegrityError:
            # Ignora se já existirem
            pass

        conn.close()

    def _executar_consulta(self, sql, parametros=(), fetch_one=False, fetch_all=False, commit=False):
        """Método utilitário para executar consultas, garantindo a parametrização e tratamento de erros."""
        conn = None
        try:
            conn = sqlite3.connect(self.db_path)
            cursor = conn.cursor()
            cursor.execute(sql, parametros)

            if commit:
                conn.commit()

            if fetch_one:
                return cursor.fetchone()
            elif fetch_all:
                return cursor.fetchall()
            else:
                return True # Sucesso na execução

        except sqlite3.IntegrityError as e:
            # Erro de integridade (ex: nome duplicado)
            print(f"Erro de Integridade do Banco de Dados: {e}")
            return False
        except sqlite3.Error as e:
            # Outros erros de SQLite
            print(f"Erro no banco de dados: {e}")
            return False

        finally:
            if conn:
                conn.close()

    # --- Métodos de Transações ---

    def adicionar_transacao(self, id_conta, id_categoria, tipo, valor, data, descricao):
        """Adiciona uma nova transação."""
        sql = """
            INSERT INTO transacoes (id_conta, id_categoria, tipo, valor, data, descricao)
            VALUES (?, ?, ?, ?, ?, ?)
        """
        parametros = (id_conta, id_categoria, tipo, valor, data, descricao)
        return self._executar_consulta(sql, parametros, commit=True)

    def buscar_todas_transacoes(self):
        """Busca todas as transações com nomes de conta e categoria."""
        sql = """
            SELECT
                t.id, cta.nome, cat.nome, t.tipo, t.valor, t.data, t.descricao
            FROM
                transacoes t
            JOIN
                contas cta ON t.id_conta = cta.id
            JOIN
                categorias cat ON t.id_categoria = cat.id
            ORDER BY
                t.data DESC
        """
        return self._executar_consulta(sql, fetch_all=True)

    def buscar_transacoes_por_categoria(self, id_categoria):
        """Busca transações filtradas por id de categoria."""
        sql = """
            SELECT
                t.id, cta.nome, cat.nome, t.tipo, t.valor, t.data, t.descricao
            FROM
                transacoes t
            JOIN
                contas cta ON t.id_conta = cta.id
            JOIN
                categorias cat ON t.id_categoria = cat.id
            WHERE
                t.id_categoria = ?
            ORDER BY
                t.data DESC
        """
        return self._executar_consulta(sql, (id_categoria,), fetch_all=True)

    # --- Métodos de Contas ---

    def buscar_contas(self):
        """Retorna todas as contas (id, nome)."""
        sql = "SELECT id, nome FROM contas ORDER BY nome"
        return self._executar_consulta(sql, fetch_all=True)

    def buscar_conta_por_nome(self, nome):
        """Retorna o id de uma conta pelo nome."""
        sql = "SELECT id FROM contas WHERE nome = ?"
        resultado = self._executar_consulta(sql, (nome,), fetch_one=True)
        return resultado[0] if resultado else None

    # --- Métodos de Categorias ---

    def adicionar_categoria(self, nome, tipo):
        """Adiciona uma nova categoria."""
        sql = "INSERT INTO categorias (nome, tipo) VALUES (?, ?)"
        parametros = (nome, tipo)
        return self._executar_consulta(sql, parametros, commit=True)

    def excluir_categoria(self, id_categoria):
        """Exclui uma categoria pelo ID."""
        sql = "DELETE FROM categorias WHERE id = ?"
        return self._executar_consulta(sql, (id_categoria,), commit=True)

    def buscar_categorias(self, tipo=None):
        """Retorna todas as categorias, opcionalmente filtradas por tipo."""
        if tipo in ('RECEITA', 'DESPESA'):
            sql = "SELECT id, nome, tipo FROM categorias WHERE tipo = ? ORDER BY nome"
            return self._executar_consulta(sql, (tipo,), fetch_all=True)
        else:
            sql = "SELECT id, nome, tipo FROM categorias ORDER BY tipo, nome"
            return self._executar_consulta(sql, fetch_all=True)

    def buscar_categoria_por_nome(self, nome):
        """Retorna o id e tipo de uma categoria pelo nome."""
        sql = "SELECT id, tipo FROM categorias WHERE nome = ?"
        resultado = self._executar_consulta(sql, (nome,), fetch_one=True)
        return resultado if resultado else None

    # --- Métodos de Relatório ---

    def calcular_saldo_total(self):
        """Calcula o saldo total (Receitas - Despesas + Saldo Inicial das Contas)."""

        # 1. Saldo Inicial Total das Contas
        sql_inicial = "SELECT SUM(saldo_inicial) FROM contas"
        saldo_inicial = self._executar_consulta(sql_inicial, fetch_one=True)
        saldo_inicial = saldo_inicial[0] if saldo_inicial and saldo_inicial[0] is not None else 0.0

        # 2. Total de Receitas
        sql_receitas = "SELECT SUM(valor) FROM transacoes WHERE tipo = 'RECEITA'"
        total_receitas = self._executar_consulta(sql_receitas, fetch_one=True)
        total_receitas = total_receitas[0] if total_receitas and total_receitas[0] is not None else 0.0

        # 3. Total de Despesas
        sql_despesas = "SELECT SUM(valor) FROM transacoes WHERE tipo = 'DESPESA'"
        total_despesas = self._executar_consulta(sql_despesas, fetch_one=True)
        total_despesas = total_despesas[0] if total_despesas and total_despesas[0] is not None else 0.0

        saldo_atual = saldo_inicial + total_receitas - total_despesas
        return saldo_atual, total_receitas, total_despesas, saldo_inicial

# --- Classes de Modelo (Representação de Dados) ---

class Categoria:
    """Representa uma Categoria (Receita ou Despesa)."""
    def __init__(self, id, nome, tipo):
        self.id = id
        self.nome = nome
        self.tipo = tipo

class Transacao:
    """Representa uma Transação Financeira."""
    def __init__(self, id, id_conta, id_categoria, tipo, valor, data, descricao):
        self.id = id
        self.id_conta = id_conta
        self.id_categoria = id_categoria
        self.tipo = tipo
        self.valor = valor
        self.data = data
        self.descricao = descricao

# --- Classe AppGUI (Interface Gráfica) ---

class AppGUI:
    """
    Interface Gráfica Principal (Tkinter).
    Responsável por construir e gerenciar a interface e manipular eventos.
    Possui uma instância de DBManager para todas as operações.
    
    ATENÇÃO: Este código não será executado no Colab. Ele é incluído para 
    demonstrar a arquitetura POO completa e a interface Tkinter solicitada.
    """
    def __init__(self, master, db_manager):
        self.master = master
        self.db = db_manager
        master.title("FinançaSimples - Gerenciamento de Finanças Pessoais")
        master.geometry("800x600")

        # Configuração de Estilo (Melhorando a aparência com ttk)
        style = ttk.Style()
        style.theme_use("clam") # Tema moderno do ttk

        # Notebook (Abas)
        self.notebook = ttk.Notebook(master)
        self.notebook.pack(pady=10, padx=10, expand=True, fill="both")

        # Criação das Abas
        self.aba_registro = ttk.Frame(self.notebook)
        self.aba_categorias = ttk.Frame(self.notebook)
        self.aba_relatorios = ttk.Frame(self.notebook)

        self.notebook.add(self.aba_registro, text="Registro de Transações")
        self.notebook.add(self.aba_categorias, text="Gerenciamento de Categorias")
        self.notebook.add(self.aba_relatorios, text="Consultas e Relatórios")

        # Inicialização dos componentes de cada aba
        self._setup_aba_registro()
        self._setup_aba_categorias()
        self._setup_aba_relatorios()

    # --- Configuração da Aba de Registro de Transações ---

    def _setup_aba_registro(self):
        frame = ttk.Frame(self.aba_registro, padding="10")
        frame.pack(fill="x", padx=10, pady=10)

        # Variáveis de controle
        self.var_tipo = tk.StringVar(value="DESPESA")
        self.var_conta = tk.StringVar()
        self.var_categoria = tk.StringVar()
        self.var_valor = tk.StringVar()
        self.var_descricao = tk.StringVar()
        self.var_data = tk.StringVar(value=datetime.now().strftime("%Y-%m-%d"))

        # Tipo (Receita/Despesa)
        ttk.Label(frame, text="Tipo:").grid(row=0, column=0, sticky="w", pady=5, padx=5)
        ttk.Radiobutton(frame, text="Receita", variable=self.var_tipo, value="RECEITA", command=self._atualizar_categorias_registro).grid(row=0, column=1, sticky="w", pady=5, padx=5)
        ttk.Radiobutton(frame, text="Despesa", variable=self.var_tipo, value="DESPESA", command=self._atualizar_categorias_registro).grid(row=0, column=2, sticky="w", pady=5, padx=5)

        # Conta
        ttk.Label(frame, text="Conta:").grid(row=1, column=0, sticky="w", pady=5, padx=5)
        self.combo_conta = ttk.Combobox(frame, textvariable=self.var_conta, state="readonly")
        self.combo_conta.grid(row=1, column=1, columnspan=2, sticky="ew", pady=5, padx=5)

        # Categoria
        ttk.Label(frame, text="Categoria:").grid(row=2, column=0, sticky="w", pady=5, padx=5)
        self.combo_categoria = ttk.Combobox(frame, textvariable=self.var_categoria, state="readonly")
        self.combo_categoria.grid(row=2, column=1, columnspan=2, sticky="ew", pady=5, padx=5)

        # Valor
        ttk.Label(frame, text="Valor (R$):").grid(row=3, column=0, sticky="w", pady=5, padx=5)
        ttk.Entry(frame, textvariable=self.var_valor).grid(row=3, column=1, columnspan=2, sticky="ew", pady=5, padx=5)

        # Data
        ttk.Label(frame, text="Data (AAAA-MM-DD):").grid(row=4, column=0, sticky="w", pady=5, padx=5)
        ttk.Entry(frame, textvariable=self.var_data).grid(row=4, column=1, columnspan=2, sticky="ew", pady=5, padx=5)

        # Descrição
        ttk.Label(frame, text="Descrição:").grid(row=5, column=0, sticky="w", pady=5, padx=5)
        ttk.Entry(frame, textvariable=self.var_descricao).grid(row=5, column=1, columnspan=2, sticky="ew", pady=5, padx=5)

        # Botão Adicionar
        ttk.Button(frame, text="Adicionar Transação", command=self._adicionar_transacao).grid(row=6, column=0, columnspan=3, pady=10)

        # Configuração de expansão
        frame.columnconfigure(1, weight=1)
        frame.columnconfigure(2, weight=1)

        # Inicializa os dados dos Comboboxes
        self._carregar_comboboxes_registro()

    def _carregar_comboboxes_registro(self):
        # Carregar Contas
        contas = self.db.buscar_contas()
        nomes_contas = [nome for id, nome in contas] if contas else []
        self.combo_conta['values'] = nomes_contas
        if nomes_contas:
            self.var_conta.set(nomes_contas[0])
        else:
            self.var_conta.set("")

        # Carregar Categorias
        self._atualizar_categorias_registro()

    def _atualizar_categorias_registro(self):
        tipo = self.var_tipo.get()
        categorias = self.db.buscar_categorias(tipo=tipo)
        nomes_categorias = [nome for id, nome, tipo in categorias] if categorias else []
        self.combo_categoria['values'] = nomes_categorias
        if nomes_categorias:
            self.var_categoria.set(nomes_categorias[0])
        else:
            self.var_categoria.set("")

    def _adicionar_transacao(self):
        try:
            # 1. Obter e validar dados
            tipo = self.var_tipo.get()
            nome_conta = self.var_conta.get()
            nome_categoria = self.var_categoria.get()
            valor_str = self.var_valor.get().replace(',', '.')
            data = self.var_data.get()
            descricao = self.var_descricao.get()

            # Validação básica
            if not nome_conta or not nome_categoria or not valor_str or not data:
                messagebox.showerror("Erro de Validação", "Todos os campos (Conta, Categoria, Valor, Data) são obrigatórios.")
                return

            try:
                valor = float(valor_str)
                if valor <= 0:
                    messagebox.showerror("Erro de Validação", "O Valor deve ser positivo.")
                    return
            except ValueError:
                messagebox.showerror("Erro de Formato", "O campo Valor deve ser um número válido.")
                return

            # 2. Buscar IDs
            id_conta = self.db.buscar_conta_por_nome(nome_conta)
            cat_info = self.db.buscar_categoria_por_nome(nome_categoria)
            id_categoria = cat_info[0] if cat_info else None

            if not id_conta or not id_categoria:
                messagebox.showerror("Erro de Dados", "Conta ou Categoria selecionada não encontrada no banco de dados. Verifique a aba 'Gerenciamento de Categorias'.")
                return

            # 3. Inserir Transação
            if self.db.adicionar_transacao(id_conta, id_categoria, tipo, valor, data, descricao):
                messagebox.showinfo("Sucesso", f"Transação de {tipo} adicionada com sucesso!")
                # Limpar formulário (exceto tipo e data)
                self.var_valor.set("")
                self.var_descricao.set("")
                # Atualizar relatórios
                self._atualizar_relatorio()
            else:
                messagebox.showerror("Erro", "Falha ao adicionar transação. Verifique o console para detalhes.")

        except Exception as e:
            messagebox.showerror("Erro Inesperado", f"Ocorreu um erro: {e}")

    # --- Configuração da Aba de Gerenciamento de Categorias ---

    def _setup_aba_categorias(self):
        # Frame de Adição
        frame_add = ttk.LabelFrame(self.aba_categorias, text="Adicionar Nova Categoria", padding="10")
        frame_add.pack(fill="x", padx=10, pady=10)

        self.var_cat_nome = tk.StringVar()
        self.var_cat_tipo = tk.StringVar(value="DESPESA")

        ttk.Label(frame_add, text="Nome:").grid(row=0, column=0, sticky="w", padx=5, pady=5)
        ttk.Entry(frame_add, textvariable=self.var_cat_nome, width=30).grid(row=0, column=1, sticky="ew", padx=5, pady=5)

        ttk.Label(frame_add, text="Tipo:").grid(row=1, column=0, sticky="w", padx=5, pady=5)
        ttk.Radiobutton(frame_add, text="Receita", variable=self.var_cat_tipo, value="RECEITA").grid(row=1, column=1, sticky="w", padx=5, pady=5)
        ttk.Radiobutton(frame_add, text="Despesa", variable=self.var_cat_tipo, value="DESPESA").grid(row=1, column=2, sticky="w", padx=5, pady=5)

        ttk.Button(frame_add, text="Adicionar", command=self._adicionar_categoria).grid(row=0, column=3, rowspan=2, padx=10, pady=5, sticky="e")
        frame_add.columnconfigure(1, weight=1)

        # Frame de Visualização
        frame_view = ttk.LabelFrame(self.aba_categorias, text="Categorias Existentes", padding="10")
        frame_view.pack(fill="both", expand=True, padx=10, pady=10)

        # Treeview
        self.tree_categorias = ttk.Treeview(frame_view, columns=("ID", "Nome", "Tipo"), show="headings")
        self.tree_categorias.heading("ID", text="ID", anchor=tk.CENTER)
        self.tree_categorias.heading("Nome", text="Nome")
        self.tree_categorias.heading("Tipo", text="Tipo")

        self.tree_categorias.column("ID", width=50, anchor=tk.CENTER)
        self.tree_categorias.column("Nome", width=200)
        self.tree_categorias.column("Tipo", width=100, anchor=tk.CENTER)

        self.tree_categorias.pack(side="left", fill="both", expand=True)

        # Scrollbar
        scrollbar = ttk.Scrollbar(frame_view, orient="vertical", command=self.tree_categorias.yview)
        self.tree_categorias.configure(yscrollcommand=scrollbar.set)
        scrollbar.pack(side="right", fill="y")

        # Botão Excluir
        ttk.Button(frame_view, text="Excluir Categoria Selecionada", command=self._excluir_categoria).pack(pady=5)

        # Inicializa a lista
        self._atualizar_lista_categorias()

    def _adicionar_categoria(self):
        nome = self.var_cat_nome.get().strip()
        tipo = self.var_cat_tipo.get()

        if not nome:
            messagebox.showerror("Erro", "O nome da categoria não pode ser vazio.")
            return

        if self.db.adicionar_categoria(nome, tipo):
            messagebox.showinfo("Sucesso", f"Categoria '{nome}' ({tipo}) adicionada com sucesso.")
            self.var_cat_nome.set("")
            self._atualizar_lista_categorias()
            self._atualizar_categorias_registro() # Atualiza o combobox de registro
            self._carregar_combobox_filtro_categorias() # Atualiza o combobox de filtro
        else:
            messagebox.showerror("Erro", "Falha ao adicionar categoria. O nome pode ser duplicado.")

    def _excluir_categoria(self):
        selected_item = self.tree_categorias.focus()
        if not selected_item:
            messagebox.showwarning("Aviso", "Selecione uma categoria para excluir.")
            return

        values = self.tree_categorias.item(selected_item, 'values')
        cat_id = values[0]
        cat_nome = values[1]

        if messagebox.askyesno("Confirmação", f"Tem certeza que deseja excluir a categoria '{cat_nome}'? \nIsso pode causar erros em transações que a utilizam."):
            if self.db.excluir_categoria(cat_id):
                messagebox.showinfo("Sucesso", f"Categoria '{cat_nome}' excluída.")
                self._atualizar_lista_categorias()
                self._atualizar_categorias_registro() # Atualiza o combobox de registro
                self._carregar_combobox_filtro_categorias() # Atualiza o combobox de filtro
            else:
                messagebox.showerror("Erro", "Falha ao excluir categoria. Provavelmente há transações associadas a ela.")

    def _atualizar_lista_categorias(self):
        # Limpa o Treeview
        for i in self.tree_categorias.get_children():
            self.tree_categorias.delete(i)

        # Insere os novos dados
        categorias = self.db.buscar_categorias()
        if categorias:
            for cat in categorias:
                # cat é (id, nome, tipo)
                self.tree_categorias.insert("", "end", values=cat)

    # --- Configuração da Aba de Relatórios ---

    def _setup_aba_relatorios(self):
        # Frame de Saldo Total
        frame_saldo = ttk.LabelFrame(self.aba_relatorios, text="Saldo Total", padding="10")
        frame_saldo.pack(fill="x", padx=10, pady=10)

        self.label_saldo = ttk.Label(frame_saldo, text="Clique em 'Calcular Saldo' para ver o resumo.", font=("Arial", 14, "bold"))
        self.label_saldo.pack(pady=10)

        ttk.Button(frame_saldo, text="Calcular Saldo", command=self._atualizar_relatorio).pack(pady=5)

        # Frame de Relatório por Categoria
        frame_cat = ttk.LabelFrame(self.aba_relatorios, text="Transações por Categoria", padding="10")
        frame_cat.pack(fill="both", expand=True, padx=10, pady=10)

        # Seleção de Categoria
        frame_filtro = ttk.Frame(frame_cat)
        frame_filtro.pack(fill="x", pady=5)

        self.var_filtro_categoria = tk.StringVar()
        ttk.Label(frame_filtro, text="Categoria:").pack(side="left", padx=5)
        self.combo_filtro_categoria = ttk.Combobox(frame_filtro, textvariable=self.var_filtro_categoria, state="readonly")
        self.combo_filtro_categoria.pack(side="left", expand=True, fill="x", padx=5)
        ttk.Button(frame_filtro, text="Ver Transações", command=self._ver_transacoes_por_categoria).pack(side="left", padx=5)

        # Treeview de Transações
        self.tree_transacoes = ttk.Treeview(frame_cat, columns=("ID", "Conta", "Categoria", "Tipo", "Valor", "Data", "Descrição"), show="headings")
        self.tree_transacoes.heading("ID", text="ID")
        self.tree_transacoes.heading("Conta", text="Conta")
        self.tree_transacoes.heading("Categoria", text="Categoria")
        self.tree_transacoes.heading("Tipo", text="Tipo")
        self.tree_transacoes.heading("Valor", text="Valor (R$)")
        self.tree_transacoes.heading("Data", text="Data")
        self.tree_transacoes.heading("Descrição", text="Descrição")

        self.tree_transacoes.column("ID", width=50, anchor=tk.CENTER)
        self.tree_transacoes.column("Conta", width=100)
        self.tree_transacoes.column("Categoria", width=100)
        self.tree_transacoes.column("Tipo", width=80, anchor=tk.CENTER)
        self.tree_transacoes.column("Valor", width=100, anchor=tk.E)
        self.tree_transacoes.column("Data", width=100, anchor=tk.CENTER)
        self.tree_transacoes.column("Descrição", width=200)

        self.tree_transacoes.pack(side="left", fill="both", expand=True)

        # Scrollbar
        scrollbar_transacoes = ttk.Scrollbar(frame_cat, orient="vertical", command=self.tree_transacoes.yview)
        self.tree_transacoes.configure(yscrollcommand=scrollbar_transacoes.set)
        scrollbar_transacoes.pack(side="right", fill="y")

        # Inicializa o filtro de categorias
        self._carregar_combobox_filtro_categorias()
        self._atualizar_relatorio() # Calcula o saldo inicial

    def _carregar_combobox_filtro_categorias(self):
        categorias = self.db.buscar_categorias()
        nomes_categorias = [nome for id, nome, tipo in categorias] if categorias else []
        self.combo_filtro_categoria['values'] = nomes_categorias
        if nomes_categorias:
            self.var_filtro_categoria.set(nomes_categorias[0])
        else:
            self.var_filtro_categoria.set("")

    def _atualizar_relatorio(self):
        """Calcula e exibe o saldo total."""
        saldo, receitas, despesas, inicial = self.db.calcular_saldo_total()

        # Função auxiliar para formatação de moeda
        def formatar_moeda(valor):
            return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

        texto_relatorio = (
            f"Saldo Inicial das Contas: {formatar_moeda(inicial)}\n"
            f"Total de Receitas: {formatar_moeda(receitas)}\n"
            f"Total de Despesas: {formatar_moeda(despesas)}\n"
            f"Saldo Total Atual: {formatar_moeda(saldo)}"
        )
        self.label_saldo.config(text=texto_relatorio)

    def _ver_transacoes_por_categoria(self):
        nome_categoria = self.var_filtro_categoria.get()

        # Limpa o Treeview
        for i in self.tree_transacoes.get_children():
            self.tree_transacoes.delete(i)

        if not nome_categoria:
            return

        cat_info = self.db.buscar_categoria_por_nome(nome_categoria)
        if not cat_info:
            messagebox.showerror("Erro", "Categoria não encontrada.")
            return

        id_categoria = cat_info[0]
        transacoes = self.db.buscar_transacoes_por_categoria(id_categoria)

        if transacoes:
            for t in transacoes:
                # t é (t.id, cta.nome, cat.nome, t.tipo, t.valor, t.data, t.descricao)
                # Formata o valor para exibição
                valor_formatado = f"{t[4]:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
                item_formatado = (t[0], t[1], t[2], t[3], valor_formatado, t[5], t[6])
                self.tree_transacoes.insert("", "end", values=item_formatado)
        else:
            # Adiciona uma linha de aviso se não houver transações
            self.tree_transacoes.insert("", "end", values=("", "Nenhuma transação", "encontrada", "para", nome_categoria, "", ""), tags=('warning',))
            self.tree_transacoes.tag_configure('warning', foreground='red')


# -------------------------------------------------------------------
# --- Bloco de Teste de Lógica para Execução no Google Colab ---
# -------------------------------------------------------------------

# Este bloco testa a lógica do DBManager, que é a parte funcional no Colab.

try:
    # Inicializa o DBManager (cria o banco e dados iniciais)
    db = DBManager()
    print("\n[SUCESSO] Banco de dados 'financas.db' inicializado com sucesso, atendendo aos requisitos de POO e SQLite.")

    # --- Teste de Adição de Transação ---
    id_conta = db.buscar_conta_por_nome("Conta Corrente")
    cat_salario = db.buscar_categoria_por_nome("Salário")
    cat_aluguel = db.buscar_categoria_por_nome("Aluguel")

    if id_conta and cat_salario and cat_aluguel:
        # Adicionar Receita
        db.adicionar_transacao(
            id_conta=id_conta,
            id_categoria=cat_salario[0],
            tipo=cat_salario[1],
            valor=3000.00,
            data="2025-10-01",
            descricao="Salário de Outubro (Teste)"
        )
        # Adicionar Despesa
        db.adicionar_transacao(
            id_conta=id_conta,
            id_categoria=cat_aluguel[0],
            tipo=cat_aluguel[1],
            valor=1500.00,
            data="2025-10-05",
            descricao="Pagamento de Aluguel (Teste)"
        )
        print("[SUCESSO] Transações de teste adicionadas ao banco de dados.")

    # --- Teste de Relatório ---
    saldo, receitas, despesas, inicial = db.calcular_saldo_total()
    print("\n--- Relatório Financeiro (Teste de Lógica) ---")
    print(f"Saldo Inicial Total das Contas: R$ {inicial:,.2f}")
    print(f"Total de Receitas: R$ {receitas:,.2f}")
    print(f"Total de Despesas: R$ {despesas:,.2f}")
    print(f"Saldo Total Atual: R$ {saldo:,.2f}")

    # --- Teste de Busca de Transações ---
    print("\n--- Todas as Transações Registradas (Teste) ---")
    transacoes = db.buscar_todas_transacoes()
    if transacoes:
        for t in transacoes:
            print(f"ID: {t[0]} | Conta: {t[1]} | Categoria: {t[2]} | Tipo: {t[3]} | Valor: R$ {t[4]:.2f} | Data: {t[5]}")
    else:
        print("Nenhuma transação encontrada.")

    # --- Execução da GUI (Apenas para demonstração da arquitetura) ---
    # O bloco abaixo será ignorado no Colab, mas é a implementação completa do Tkinter.
    # Se for executado em um ambiente local, a GUI será iniciada.
    # root = tk.Tk()
    # app = AppGUI(root, db)
    # root.mainloop()

except Exception as e:
    print(f"\n[ERRO FATAL] Falha na execução do teste de lógica: {e}")
